# Urchin v1 Dataset Review and Seeded Split

This notebook performs a local dataset review for YOLO object-detection format and then creates a deterministic seeded split.

Key features:
- Dataset summary: image count, label coverage, per-class object counts, and basic annotation QA checks
- Flexible filtering: exclude classes, enforce min/max objects per image, handle empty labels
- Deterministic split: train/val/test using a fixed seed
- Export-ready output: copied images/labels plus CSV manifests and summaries

In [66]:
from __future__ import annotations

import json
import math
import random
import shutil
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

In [ ]:
# Paths
DATASET_ROOT = Path(r"C:\Users\michael.akridge\Desktop\20260713\optics-si-special-projects\urchins\multi-class\v1\dataset")
IMAGES_DIR = DATASET_ROOT / 'images'
LABELS_DIR = DATASET_ROOT / 'labels'
CLASSES_PATH = DATASET_ROOT / 'classes.txt'
NOTES_PATH = DATASET_ROOT / 'notes.json'

OUTPUT_ROOT = DATASET_ROOT.parent / 'dataset_review_and_split'
REVIEW_OUTPUT_DIR = OUTPUT_ROOT / 'review'
SPLIT_OUTPUT_DIR = OUTPUT_ROOT / 'split'

# Review behavior
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
LABEL_EXTENSION = '.txt'
MATCH_STEM_MODE = 'canonical_tail'  # 'exact', 'normalized', or 'canonical_tail'

# Optional in-place reconciliation
APPLY_INPLACE_RENAME = False
RENAME_TARGET = 'labels'  # 'labels', 'images', or 'both'
RENAME_BOTH_CANONICAL = 'image'  # 'image' or 'label'
DRY_RUN_RENAME = True

# Filtering behavior
EXCLUDE_CLASS_NAMES = []  # Example: ['ECMA', 'PAGR']
EXCLUDE_CLASS_IDS = []    # Example: [2, 7]
REQUIRE_LABEL_FOR_IMAGE = True
DROP_IMAGES_WITH_NO_OBJECTS = True
MIN_OBJECTS_PER_IMAGE = None  # Example: 1
MAX_OBJECTS_PER_IMAGE = None  # Example: 50

# Pre-split class balancing behavior
PRE_SPLIT_EXCLUDE_BELOW_ANNOTATIONS = 50  # Set to None to disable automatic low-count class exclusion
ENABLE_PRE_SPLIT_UNDERSAMPLE = False
PRE_SPLIT_UNDERSAMPLE_MAX_ANNOTATIONS = None  # Example: 1200
PRE_SPLIT_UNDERSAMPLE_REFERENCE = 'median'  # 'min', 'median', or 'p75'

# Split behavior
# note for michael make sysdate 
RANDOM_SEED = 20260713
SPLIT_RATIOS = {'train': 0.80, 'val': 0.15, 'test': 0.10}
STRATIFY_BY_PRIMARY_CLASS = True

# File output behavior
CLEAN_OUTPUT_BEFORE_WRITE = True
COPY_IMAGES = True
WRITE_FILTERED_LABELS = True
WRITE_EMPTY_LABEL_FILES = False

# Post-processing for training compatibility
FORCE_LABEL_FILE_FOR_EVERY_IMAGE = True
REMOVE_ORPHAN_LABEL_FILES = True
STRICT_PAIR_CHECK = True

print({
    'dataset_root': str(DATASET_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'random_seed': RANDOM_SEED,
    'split_ratios': SPLIT_RATIOS,
    'exclude_class_names': EXCLUDE_CLASS_NAMES,
    'exclude_class_ids': EXCLUDE_CLASS_IDS,
    'match_stem_mode': MATCH_STEM_MODE,
    'apply_inplace_rename': APPLY_INPLACE_RENAME,
    'rename_target': RENAME_TARGET,
    'dry_run_rename': DRY_RUN_RENAME,
    'pre_split_exclude_below_annotations': PRE_SPLIT_EXCLUDE_BELOW_ANNOTATIONS,
    'enable_pre_split_undersample': ENABLE_PRE_SPLIT_UNDERSAMPLE,
    'pre_split_undersample_max_annotations': PRE_SPLIT_UNDERSAMPLE_MAX_ANNOTATIONS,
    'pre_split_undersample_reference': PRE_SPLIT_UNDERSAMPLE_REFERENCE,
    'force_label_file_for_every_image': FORCE_LABEL_FILE_FOR_EVERY_IMAGE,
    'remove_orphan_label_files': REMOVE_ORPHAN_LABEL_FILES,
    'strict_pair_check': STRICT_PAIR_CHECK,
})

{'dataset_root': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset', 'output_root': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset_review_and_split', 'random_seed': 20260713, 'split_ratios': {'train': 0.8, 'val': 0.1, 'test': 0.1}, 'exclude_class_names': [], 'exclude_class_ids': [], 'match_stem_mode': 'canonical_tail', 'apply_inplace_rename': False, 'rename_target': 'labels', 'dry_run_rename': True, 'pre_split_exclude_below_annotations': 50, 'enable_pre_split_undersample': False, 'pre_split_undersample_max_annotations': None, 'pre_split_undersample_reference': 'median', 'force_label_file_for_every_image': True, 'remove_orphan_label_files': True, 'strict_pair_check': True}


In [68]:
@dataclass
class ImageRecord:
    image_name: str
    image_path: Path
    label_path: Path | None
    has_label_file: bool
    match_mode: str
    raw_object_count: int
    kept_object_count: int
    excluded_object_count: int
    kept_label_lines: list[str]
    raw_class_ids: list[int]
    kept_class_ids: list[int]
    exclusion_reason: str


def load_classes(classes_path: Path, notes_path: Path) -> list[str]:
    classes = []
    if classes_path.exists():
        classes = [line.strip() for line in classes_path.read_text(encoding='utf-8').splitlines() if line.strip()]

    if not classes and notes_path.exists():
        data = json.loads(notes_path.read_text(encoding='utf-8'))
        categories = data.get('categories', [])
        categories = sorted(categories, key=lambda c: c.get('id', 0))
        classes = [c.get('name', f"class_{c.get('id', 0)}") for c in categories]

    if not classes:
        raise FileNotFoundError('No classes found. Expected classes.txt or notes.json categories.')

    return classes


def list_images(images_dir: Path, exts: set[str]) -> list[Path]:
    return sorted([p for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in exts])


def list_labels(labels_dir: Path, label_extension: str = '.txt') -> list[Path]:
    return sorted([p for p in labels_dir.iterdir() if p.is_file() and p.suffix.lower() == label_extension.lower()])


def normalize_stem(stem: str) -> str:
    import re
    return re.sub(r'[^a-z0-9]+', '', stem.lower())


def canonical_tail_stem(stem: str) -> str:
    import re
    canonical = re.sub(r'^[0-9a-fA-F]{8}__', '', stem)
    canonical = re.sub(r'^[0-9a-fA-F]{8}-', '', canonical)
    return canonical.lower()


def make_match_key(stem: str, mode: str) -> str:
    if mode == 'exact':
        return stem
    if mode == 'normalized':
        return normalize_stem(stem)
    return canonical_tail_stem(stem)


def build_stem_lookup(paths: list[Path], mode: str) -> dict[str, list[Path]]:
    lookup: dict[str, list[Path]] = {}
    for path in paths:
        key = make_match_key(path.stem, mode)
        lookup.setdefault(key, []).append(path)
    return lookup


def primary_class_id_and_count(class_ids: list[int]) -> tuple[int | None, int]:
    if not class_ids:
        return None, 0
    top_class_id, top_count = Counter(class_ids).most_common(1)[0]
    return int(top_class_id), int(top_count)


def resolve_pre_split_annotation_cap(
    class_annotation_counts: dict[int, int],
    explicit_max: int | None,
    reference: str,
 ) -> int | None:
    if explicit_max is not None:
        return int(explicit_max)

    values = sorted([int(v) for v in class_annotation_counts.values() if int(v) > 0])
    if not values:
        return None

    if reference == 'min':
        return values[0]
    if reference == 'p75':
        idx = max(0, int(round(0.75 * (len(values) - 1))))
        return values[idx]

    mid = len(values) // 2
    if len(values) % 2 == 1:
        return values[mid]
    return int(round((values[mid - 1] + values[mid]) / 2))


def build_rename_plan(
    matched_pairs: list[tuple[Path, Path]],
    rename_target: str,
    rename_both_canonical: str,
) -> list[tuple[Path, Path, str]]:
    plan: list[tuple[Path, Path, str]] = []

    for image_path, label_path in matched_pairs:
        if image_path.stem == label_path.stem:
            continue

        if rename_target == 'labels':
            target_stem = image_path.stem
            target_label = label_path.with_name(f'{target_stem}{label_path.suffix}')
            if target_label != label_path:
                plan.append((label_path, target_label, 'label'))
        elif rename_target == 'images':
            target_stem = label_path.stem
            target_image = image_path.with_name(f'{target_stem}{image_path.suffix}')
            if target_image != image_path:
                plan.append((image_path, target_image, 'image'))
        else:
            target_stem = image_path.stem if rename_both_canonical == 'image' else label_path.stem
            target_label = label_path.with_name(f'{target_stem}{label_path.suffix}')
            target_image = image_path.with_name(f'{target_stem}{image_path.suffix}')
            if target_label != label_path:
                plan.append((label_path, target_label, 'label'))
            if target_image != image_path:
                plan.append((image_path, target_image, 'image'))

    dedup = {}
    for src, dst, kind in plan:
        dedup[(str(src), str(dst), kind)] = (src, dst, kind)
    return list(dedup.values())


def validate_rename_plan(plan: list[tuple[Path, Path, str]]) -> None:
    srcs = [str(src) for src, _, _ in plan]
    dsts = [str(dst) for _, dst, _ in plan]

    if len(dsts) != len(set(dsts)):
        raise RuntimeError('Rename plan has conflicting destination paths.')

    src_set = set(srcs)
    for _, dst, _ in plan:
        if dst.exists() and str(dst) not in src_set:
            raise RuntimeError(f'Rename destination already exists: {dst}')


def parse_yolo_label_line(line: str) -> tuple[int, float, float, float, float]:
    parts = line.strip().split()
    if len(parts) != 5:
        raise ValueError(f'Expected 5 fields, got {len(parts)}: {line.strip()}')

    class_id = int(float(parts[0]))
    cx, cy, w, h = map(float, parts[1:])
    return class_id, cx, cy, w, h


def format_yolo_label_line(class_id: int, cx: float, cy: float, w: float, h: float) -> str:
    return f"{class_id} {cx:.12g} {cy:.12g} {w:.12g} {h:.12g}"


def split_counts(total: int, ratios: dict[str, float]) -> dict[str, int]:
    base = {k: int(math.floor(total * r)) for k, r in ratios.items()}
    remainder = total - sum(base.values())

    if remainder > 0:
        order = sorted(ratios.items(), key=lambda kv: kv[1], reverse=True)
        for i in range(remainder):
            base[order[i % len(order)][0]] += 1

    return base


def ensure_ratios(ratios: dict[str, float]) -> None:
    s = sum(ratios.values())
    if abs(s - 1.0) > 1e-9:
        raise ValueError(f'SPLIT_RATIOS must sum to 1.0, got {s}')

In [69]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset root does not exist: {DATASET_ROOT}')
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f'Images folder not found: {IMAGES_DIR}')
if not LABELS_DIR.exists():
    raise FileNotFoundError(f'Labels folder not found: {LABELS_DIR}')

if MATCH_STEM_MODE not in {'exact', 'normalized', 'canonical_tail'}:
    raise ValueError(f'Unsupported MATCH_STEM_MODE: {MATCH_STEM_MODE}')
if RENAME_TARGET not in {'labels', 'images', 'both'}:
    raise ValueError(f'Unsupported RENAME_TARGET: {RENAME_TARGET}')
if RENAME_BOTH_CANONICAL not in {'image', 'label'}:
    raise ValueError(f'Unsupported RENAME_BOTH_CANONICAL: {RENAME_BOTH_CANONICAL}')
if PRE_SPLIT_UNDERSAMPLE_REFERENCE not in {'min', 'median', 'p75'}:
    raise ValueError(f'Unsupported PRE_SPLIT_UNDERSAMPLE_REFERENCE: {PRE_SPLIT_UNDERSAMPLE_REFERENCE}')

ensure_ratios(SPLIT_RATIOS)
classes = load_classes(CLASSES_PATH, NOTES_PATH)
class_name_by_id = {i: name for i, name in enumerate(classes)}
class_id_by_name = {name: i for i, name in class_name_by_id.items()}

unknown_excludes = [name for name in EXCLUDE_CLASS_NAMES if name not in class_id_by_name]
if unknown_excludes:
    raise ValueError(f'Unknown class names in EXCLUDE_CLASS_NAMES: {unknown_excludes}')

excluded_class_ids = set(EXCLUDE_CLASS_IDS) | {class_id_by_name[name] for name in EXCLUDE_CLASS_NAMES}
bad_exclude_ids = [cid for cid in excluded_class_ids if cid not in class_name_by_id]
if bad_exclude_ids:
    raise ValueError(f'Unknown class ids in exclusion list: {bad_exclude_ids}')

image_paths = list_images(IMAGES_DIR, IMAGE_EXTENSIONS)
label_paths = list_labels(LABELS_DIR, LABEL_EXTENSION)

def collect_matching(image_paths_local: list[Path], label_paths_local: list[Path]) -> tuple[list[tuple[Path, Path]], list[dict], list[Path]]:
    label_lookup_local = build_stem_lookup(label_paths_local, MATCH_STEM_MODE)
    matched_pairs_local: list[tuple[Path, Path]] = []
    matching_rows_local = []
    used_label_paths_local: set[str] = set()

    for image_path in image_paths_local:
        key = make_match_key(image_path.stem, MATCH_STEM_MODE)
        candidates = label_lookup_local.get(key, [])

        if len(candidates) == 1:
            label_path = candidates[0]
            matched_pairs_local.append((image_path, label_path))
            used_label_paths_local.add(str(label_path))
            status = 'matched'
            mode = 'exact' if image_path.stem == label_path.stem else MATCH_STEM_MODE
        elif len(candidates) == 0:
            label_path = None
            status = 'missing_label'
            mode = 'none'
        else:
            label_path = None
            status = 'ambiguous_label_match'
            mode = 'ambiguous'

        matching_rows_local.append({
            'image_name': image_path.name,
            'image_path': str(image_path),
            'match_status': status,
            'match_mode': mode,
            'matched_label_path': str(label_path) if label_path else '',
            'candidate_count': len(candidates),
        })

    orphan_label_paths_local = [p for p in label_paths_local if str(p) not in used_label_paths_local]
    return matched_pairs_local, matching_rows_local, orphan_label_paths_local


matched_pairs, matching_rows, orphan_label_paths = collect_matching(image_paths, label_paths)

rename_plan = build_rename_plan(matched_pairs, RENAME_TARGET, RENAME_BOTH_CANONICAL)
rename_plan_df = pd.DataFrame([
    {'kind': kind, 'source_path': str(src), 'target_path': str(dst)}
    for src, dst, kind in rename_plan
])

if APPLY_INPLACE_RENAME and rename_plan:
    validate_rename_plan(rename_plan)
    if not DRY_RUN_RENAME:
        for src, dst, _ in sorted(rename_plan, key=lambda item: (item[2], str(item[0]))):
            src.rename(dst)

        image_paths = list_images(IMAGES_DIR, IMAGE_EXTENSIONS)
        label_paths = list_labels(LABELS_DIR, LABEL_EXTENSION)
        matched_pairs, matching_rows, orphan_label_paths = collect_matching(image_paths, label_paths)

records: list[ImageRecord] = []
qa_counters = Counter()
raw_object_counter = Counter()
kept_object_counter = Counter()

matching_df = pd.DataFrame(matching_rows)
label_path_by_image_name = {Path(row['image_name']).name: Path(row['matched_label_path']) for row in matching_rows if row['matched_label_path']}
match_mode_by_image_name = {Path(row['image_name']).name: row['match_mode'] for row in matching_rows}

for image_path in image_paths:
    image_name = image_path.name
    label_path = label_path_by_image_name.get(image_name)
    has_label_file = label_path is not None and label_path.exists()
    match_mode = match_mode_by_image_name.get(image_name, 'none')

    raw_lines = []
    if has_label_file:
        raw_lines = [ln.strip() for ln in label_path.read_text(encoding='utf-8').splitlines() if ln.strip()]

    parsed_rows = []
    for line in raw_lines:
        try:
            class_id, cx, cy, w, h = parse_yolo_label_line(line)
            parsed_rows.append((class_id, cx, cy, w, h))

            if class_id not in class_name_by_id:
                qa_counters['unknown_class_id_rows'] += 1
            if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                qa_counters['out_of_bounds_rows'] += 1
            if w <= 0 or h <= 0:
                qa_counters['non_positive_box_rows'] += 1
        except Exception:
            qa_counters['malformed_label_rows'] += 1

    raw_class_ids = [r[0] for r in parsed_rows]
    for cid in raw_class_ids:
        raw_object_counter[cid] += 1

    kept_rows = [r for r in parsed_rows if r[0] not in excluded_class_ids]
    kept_class_ids = [r[0] for r in kept_rows]
    for cid in kept_class_ids:
        kept_object_counter[cid] += 1

    kept_label_lines = [format_yolo_label_line(*row) for row in kept_rows]

    exclusion_reason = ''
    kept_count = len(kept_rows)

    if REQUIRE_LABEL_FOR_IMAGE and not has_label_file:
        exclusion_reason = 'missing_label_file'
        if match_mode == 'ambiguous':
            exclusion_reason = 'ambiguous_label_match'
    elif DROP_IMAGES_WITH_NO_OBJECTS and kept_count == 0:
        exclusion_reason = 'no_objects_after_filter'
    elif MIN_OBJECTS_PER_IMAGE is not None and kept_count < MIN_OBJECTS_PER_IMAGE:
        exclusion_reason = 'below_min_objects'
    elif MAX_OBJECTS_PER_IMAGE is not None and kept_count > MAX_OBJECTS_PER_IMAGE:
        exclusion_reason = 'above_max_objects'

    records.append(ImageRecord(
        image_name=image_name,
        image_path=image_path,
        label_path=label_path,
        has_label_file=has_label_file,
        match_mode=match_mode,
        raw_object_count=len(parsed_rows),
        kept_object_count=kept_count,
        excluded_object_count=len(parsed_rows) - kept_count,
        kept_label_lines=kept_label_lines,
        raw_class_ids=raw_class_ids,
        kept_class_ids=kept_class_ids,
        exclusion_reason=exclusion_reason,
    ))

candidate_records_before_pre_split = [r for r in records if r.exclusion_reason == '']
pre_split_annotations_before = Counter()
pre_split_primary_images_before = Counter()
for rec in candidate_records_before_pre_split:
    for cid in rec.kept_class_ids:
        pre_split_annotations_before[cid] += 1
    primary_cid, _ = primary_class_id_and_count(rec.kept_class_ids)
    if primary_cid is not None:
        pre_split_primary_images_before[primary_cid] += 1

pre_split_auto_excluded_class_ids = set()
if PRE_SPLIT_EXCLUDE_BELOW_ANNOTATIONS is not None:
    threshold = int(PRE_SPLIT_EXCLUDE_BELOW_ANNOTATIONS)
    pre_split_auto_excluded_class_ids = {
        cid for cid, count in pre_split_annotations_before.items() if int(count) < threshold
    }

effective_excluded_class_ids = set(excluded_class_ids) | set(pre_split_auto_excluded_class_ids)

if pre_split_auto_excluded_class_ids:
    for rec in records:
        has_label_file = rec.label_path is not None and rec.label_path.exists()
        rec.has_label_file = has_label_file
        if not has_label_file:
            continue

        raw_lines = [ln.strip() for ln in rec.label_path.read_text(encoding='utf-8').splitlines() if ln.strip()]
        parsed_rows = []
        for line in raw_lines:
            try:
                parsed_rows.append(parse_yolo_label_line(line))
            except Exception:
                continue

        kept_rows = [row for row in parsed_rows if row[0] not in effective_excluded_class_ids]
        rec.kept_class_ids = [row[0] for row in kept_rows]
        rec.kept_object_count = len(kept_rows)
        rec.excluded_object_count = len(parsed_rows) - rec.kept_object_count
        rec.kept_label_lines = [format_yolo_label_line(*row) for row in kept_rows]

        rec.exclusion_reason = ''
        if REQUIRE_LABEL_FOR_IMAGE and not rec.has_label_file:
            rec.exclusion_reason = 'missing_label_file'
        elif DROP_IMAGES_WITH_NO_OBJECTS and rec.kept_object_count == 0:
            rec.exclusion_reason = 'no_objects_after_filter'
        elif MIN_OBJECTS_PER_IMAGE is not None and rec.kept_object_count < MIN_OBJECTS_PER_IMAGE:
            rec.exclusion_reason = 'below_min_objects'
        elif MAX_OBJECTS_PER_IMAGE is not None and rec.kept_object_count > MAX_OBJECTS_PER_IMAGE:
            rec.exclusion_reason = 'above_max_objects'

pre_split_undersample_cap = None
pre_split_undersample_actions = []

if ENABLE_PRE_SPLIT_UNDERSAMPLE:
    candidate_records_for_undersample = [r for r in records if r.exclusion_reason == '']
    class_annotation_counts_for_cap = Counter()
    for rec in candidate_records_for_undersample:
        for cid in rec.kept_class_ids:
            class_annotation_counts_for_cap[cid] += 1

    pre_split_undersample_cap = resolve_pre_split_annotation_cap(
        dict(class_annotation_counts_for_cap),
        PRE_SPLIT_UNDERSAMPLE_MAX_ANNOTATIONS,
        PRE_SPLIT_UNDERSAMPLE_REFERENCE,
    )

    if pre_split_undersample_cap is not None and pre_split_undersample_cap > 0:
        primary_groups: dict[int, list[tuple[ImageRecord, int]]] = {}
        for rec in candidate_records_for_undersample:
            primary_cid, primary_count = primary_class_id_and_count(rec.kept_class_ids)
            if primary_cid is None:
                continue
            primary_groups.setdefault(primary_cid, []).append((rec, primary_count))

        for class_id, rec_list in sorted(primary_groups.items()):
            current_annotations = sum(primary_count for _, primary_count in rec_list)
            if current_annotations <= pre_split_undersample_cap:
                continue

            local_rng = random.Random(RANDOM_SEED + int(class_id))
            shuffled = sorted(rec_list, key=lambda item: item[0].image_name)
            local_rng.shuffle(shuffled)

            kept_local: list[ImageRecord] = []
            kept_annotations = 0
            for rec, primary_count in shuffled:
                if kept_annotations + primary_count <= pre_split_undersample_cap or not kept_local:
                    kept_local.append(rec)
                    kept_annotations += primary_count

            kept_names = {rec.image_name for rec in kept_local}
            dropped = 0
            dropped_annotations = 0
            for rec, primary_count in rec_list:
                if rec.image_name not in kept_names:
                    rec.exclusion_reason = 'undersampled_primary_class'
                    dropped += 1
                    dropped_annotations += primary_count

            pre_split_undersample_actions.append({
                'class_id': int(class_id),
                'class_name': class_name_by_id.get(class_id, f'UNKNOWN_{class_id}'),
                'annotations_before': int(current_annotations),
                'annotations_target_cap': int(pre_split_undersample_cap),
                'images_before': int(len(rec_list)),
                'images_dropped': int(dropped),
                'annotations_dropped': int(dropped_annotations),
            })

kept_object_counter = Counter()
for rec in records:
    if rec.exclusion_reason == '':
        for cid in rec.kept_class_ids:
            kept_object_counter[cid] += 1

kept_records = [r for r in records if r.exclusion_reason == '']
excluded_records = [r for r in records if r.exclusion_reason != '']

pre_split_annotations_after = Counter()
pre_split_primary_images_after = Counter()
pre_split_images_with_class_after = Counter()
for rec in kept_records:
    for cid in rec.kept_class_ids:
        pre_split_annotations_after[cid] += 1
    for cid in set(rec.kept_class_ids):
        pre_split_images_with_class_after[cid] += 1
    primary_cid, _ = primary_class_id_and_count(rec.kept_class_ids)
    if primary_cid is not None:
        pre_split_primary_images_after[primary_cid] += 1

pre_split_class_summary_df = pd.DataFrame([
    {
        'class_id': cid,
        'class_name': class_name_by_id.get(cid, f'UNKNOWN_{cid}'),
        'annotations_before_pre_split_rules': int(pre_split_annotations_before.get(cid, 0)),
        'annotations_after_pre_split_rules': int(pre_split_annotations_after.get(cid, 0)),
        'images_with_class_after_pre_split_rules': int(pre_split_images_with_class_after.get(cid, 0)),
        'primary_images_before_pre_split_rules': int(pre_split_primary_images_before.get(cid, 0)),
        'primary_images_after_pre_split_rules': int(pre_split_primary_images_after.get(cid, 0)),
        'auto_excluded_low_annotation': bool(cid in pre_split_auto_excluded_class_ids),
    }
    for cid in sorted(class_name_by_id.keys())
])
pre_split_class_summary_df['included_in_final_training_set'] = pre_split_class_summary_df['annotations_after_pre_split_rules'] > 0

final_class_ids = sorted([cid for cid, count in pre_split_annotations_after.items() if int(count) > 0])
final_class_names = [class_name_by_id[cid] for cid in final_class_ids]
final_class_id_remap = {old_id: new_id for new_id, old_id in enumerate(final_class_ids)}
final_class_id_unmap = {new_id: old_id for old_id, new_id in final_class_id_remap.items()}

if not final_class_ids:
    raise RuntimeError('No classes remain after filtering/pre-split rules. Lower your exclusions or thresholds.')

pre_split_undersample_actions_df = pd.DataFrame(pre_split_undersample_actions)

images_df = pd.DataFrame([
    {
        'image_name': r.image_name,
        'image_path': str(r.image_path),
        'label_path': str(r.label_path) if r.label_path else '',
        'has_label_file': r.has_label_file,
        'match_mode': r.match_mode,
        'raw_object_count': r.raw_object_count,
        'kept_object_count': r.kept_object_count,
        'excluded_object_count': r.excluded_object_count,
        'is_kept': r.exclusion_reason == '',
        'exclusion_reason': r.exclusion_reason,
    }
    for r in records
])

print({
    'classes_total': len(classes),
    'images_total': len(records),
    'images_kept': len(kept_records),
    'images_excluded': len(excluded_records),
    'orphan_labels': len(orphan_label_paths),
    'rename_plan_count': len(rename_plan),
    'manual_excluded_class_ids': sorted(excluded_class_ids),
    'auto_excluded_class_ids': sorted(pre_split_auto_excluded_class_ids),
    'effective_excluded_class_ids': sorted(effective_excluded_class_ids),
    'final_class_ids': final_class_ids,
    'final_class_names': final_class_names,
    'undersample_enabled': ENABLE_PRE_SPLIT_UNDERSAMPLE,
    'undersample_cap_annotations': pre_split_undersample_cap,
})

{'classes_total': 9, 'images_total': 983, 'images_kept': 787, 'images_excluded': 196, 'orphan_labels': 8, 'rename_plan_count': 969, 'manual_excluded_class_ids': [], 'auto_excluded_class_ids': [0], 'effective_excluded_class_ids': [0], 'final_class_ids': [1, 2, 3, 4, 6, 8], 'final_class_names': ['DISP', 'ECMA', 'ECST', 'ECTH', 'HEMA', 'TRGR'], 'undersample_enabled': False, 'undersample_cap_annotations': None}


In [70]:
raw_by_class_df = pd.DataFrame([
    {
        'class_id': cid,
        'class_name': class_name_by_id.get(cid, f'UNKNOWN_{cid}'),
        'raw_object_count': cnt,
    }
    for cid, cnt in sorted(raw_object_counter.items())
])

kept_by_class_df = pd.DataFrame([
    {
        'class_id': cid,
        'class_name': class_name_by_id.get(cid, f'UNKNOWN_{cid}'),
        'kept_object_count': cnt,
    }
    for cid, cnt in sorted(kept_object_counter.items())
])

if raw_by_class_df.empty:
    raw_by_class_df = pd.DataFrame(columns=['class_id', 'class_name', 'raw_object_count'])
if kept_by_class_df.empty:
    kept_by_class_df = pd.DataFrame(columns=['class_id', 'class_name', 'kept_object_count'])

by_class_df = raw_by_class_df.merge(kept_by_class_df, on=['class_id', 'class_name'], how='outer').fillna(0)
by_class_df['raw_object_count'] = by_class_df['raw_object_count'].astype(int)
by_class_df['kept_object_count'] = by_class_df['kept_object_count'].astype(int)
by_class_df = by_class_df.sort_values(['class_id']).reset_index(drop=True)

exclusion_reasons_df = (
    images_df[images_df['is_kept'] == False]
    .groupby('exclusion_reason')
    .size()
    .rename('image_count')
    .reset_index()
)

qa_df = pd.DataFrame([{
    'qa_issue': key,
    'count': value,
} for key, value in sorted(qa_counters.items())])

matching_summary_df = (
    matching_df.groupby(['match_status', 'match_mode']).size().rename('image_count').reset_index()
    if not matching_df.empty
    else pd.DataFrame(columns=['match_status', 'match_mode', 'image_count'])
)

orphan_labels_df = pd.DataFrame([{'label_path': str(path), 'label_name': path.name} for path in orphan_label_paths])

display(pre_split_class_summary_df.sort_values('annotations_before_pre_split_rules', ascending=False).reset_index(drop=True))
display(pre_split_undersample_actions_df)
display(by_class_df)
display(matching_summary_df)
display(exclusion_reasons_df)
display(qa_df)
display(rename_plan_df.head(50))
display(orphan_labels_df.head(50))
display(images_df.head(20))

,class_id,class_name,annotations_before_pre_split_rules,annotations_after_pre_split_rules,images_with_class_after_pre_split_rules,primary_images_before_pre_split_rules,primary_images_after_pre_split_rules,auto_excluded_low_annotation,included_in_final_training_set
0,3,ECST,2005,2005,247,223,223,False,True
1,2,ECMA,1259,1259,328,221,221,False,True
2,1,DISP,479,479,62,57,57,False,True
3,8,TRGR,318,318,144,84,84,False,True
4,6,HEMA,290,290,149,74,74,False,True
5,4,ECTH,265,265,200,128,128,False,True
6,0,CHGI,3,0,0,0,0,True,False
7,5,EUME,0,0,0,0,0,False,False
8,7,PAGR,0,0,0,0,0,False,False


""


,class_id,class_name,raw_object_count,kept_object_count
0,0,CHGI,3,0
1,1,DISP,479,479
2,2,ECMA,1259,1259
3,3,ECST,2005,2005
4,4,ECTH,265,265
5,6,HEMA,290,290
6,8,TRGR,318,318


,match_status,match_mode,image_count
0,ambiguous_label_match,ambiguous,8
1,matched,canonical_tail,969
2,missing_label,none,6


,exclusion_reason,image_count
0,ambiguous_label_match,8
1,missing_label_file,6
2,no_objects_after_filter,182


""


,kind,source_path,target_path
0,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
1,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
2,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
3,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
4,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
5,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
6,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
7,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
8,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...
9,label,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...


,label_path,label_name
0,C:\Users\michael.akridge\Desktop\20260713\opti...,0abb088c__b45fbe67-IMG_0632.txt
1,C:\Users\michael.akridge\Desktop\20260713\opti...,49118583__5dd69c65-IMG_0498.txt
2,C:\Users\michael.akridge\Desktop\20260713\opti...,4e0fc16f__7723cf88-IMG_3467.txt
3,C:\Users\michael.akridge\Desktop\20260713\opti...,662a5bc3__81e64351-IMG_0498.txt
4,C:\Users\michael.akridge\Desktop\20260713\opti...,9028ea07__IMG_3467.txt
5,C:\Users\michael.akridge\Desktop\20260713\opti...,c1a968f1__a41eb15c-HAW-3774_2019_A_18.txt
6,C:\Users\michael.akridge\Desktop\20260713\opti...,e082004a__90638ad3-HAW-3774_2019_A_18.txt
7,C:\Users\michael.akridge\Desktop\20260713\opti...,fee9f5af__5cddcfb5-IMG_0632.txt


,image_name,image_path,label_path,has_label_file,match_mode,raw_object_count,kept_object_count,excluded_object_count,is_kept,exclusion_reason
0,001448b8-OCC-KUR-013_2024_04.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,6,6,0,True,
1,0046f43e-IMG_0414.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,3,3,0,True,
2,00797fee-OCC-MAI-016_2024_17.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,6,6,0,True,
3,007ccaaf-OCC-LAN-004_2024_25.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,0,0,0,False,no_objects_after_filter
4,008524c0-IMG_3464.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,18,18,0,True,
5,0106a423-OCC-MAI-018_2024_19.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,3,3,0,True,
6,01343155-OCC-MAI-005_2024_19.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,4,4,0,True,
7,01582c60-HAW-4287_2019_A_21.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,4,4,0,True,
8,016d7d68-KA2510_A-01_IMG_5329.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,0,0,0,False,no_objects_after_filter
9,0185b3ec-HAW-3725_2019_A_22.JPG,C:\Users\michael.akridge\Desktop\20260713\opti...,C:\Users\michael.akridge\Desktop\20260713\opti...,True,canonical_tail,8,8,0,True,


In [71]:
REVIEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
review_charts_dir = REVIEW_OUTPUT_DIR / 'charts'
review_charts_dir.mkdir(parents=True, exist_ok=True)

def save_no_data_chart(path: Path, title: str, message: str = 'No data') -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, message, ha='center', va='center', fontsize=12)
    ax.set_title(title)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_bar_chart(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    path: Path,
    sort_desc: bool = False,
    rotate: int = 45,
) -> None:
    if df.empty or x_col not in df.columns or y_col not in df.columns:
        save_no_data_chart(path, title)
        return

    plot_df = df[[x_col, y_col]].copy()
    plot_df = plot_df.dropna()
    if plot_df.empty:
        save_no_data_chart(path, title)
        return

    if sort_desc:
        plot_df = plot_df.sort_values(y_col, ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df[x_col].astype(str), plot_df[y_col].astype(float))
    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.tick_params(axis='x', labelrotation=rotate)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_grouped_bar(
    df: pd.DataFrame,
    index_col: str,
    value_cols: list[str],
    title: str,
    path: Path,
    rotate: int = 45,
) -> None:
    if df.empty or index_col not in df.columns or any(col not in df.columns for col in value_cols):
        save_no_data_chart(path, title)
        return

    plot_df = df[[index_col, *value_cols]].copy()
    plot_df = plot_df.dropna(how='all', subset=value_cols)
    if plot_df.empty:
        save_no_data_chart(path, title)
        return

    plot_df = plot_df.set_index(index_col)
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_df.plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.set_xlabel(index_col)
    ax.set_ylabel('count')
    ax.tick_params(axis='x', labelrotation=rotate)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_hist(values: pd.Series, title: str, path: Path, bins: int = 30) -> None:
    numeric = pd.to_numeric(values, errors='coerce').dropna()
    if numeric.empty:
        save_no_data_chart(path, title)
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(numeric, bins=bins)
    ax.set_title(title)
    ax.set_xlabel('value')
    ax.set_ylabel('frequency')
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

images_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_images.csv', index=False)
by_class_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_objects_by_class.csv', index=False)
matching_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_matching.csv', index=False)
rename_plan_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_rename_plan.csv', index=False)
pre_split_class_summary_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_pre_split_class_summary.csv', index=False)
class_boxes_images_df = pre_split_class_summary_df[[
    'class_id',
    'class_name',
    'annotations_after_pre_split_rules',
    'images_with_class_after_pre_split_rules',
]].rename(columns={
    'annotations_after_pre_split_rules': 'annotations_boxes',
    'images_with_class_after_pre_split_rules': 'images_with_class',
})
class_boxes_images_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_class_boxes_images.csv', index=False)

if not pre_split_undersample_actions_df.empty:
    pre_split_undersample_actions_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_pre_split_undersample_actions.csv', index=False)
if not orphan_labels_df.empty:
    orphan_labels_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_orphan_labels.csv', index=False)
if not matching_summary_df.empty:
    matching_summary_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_matching_summary.csv', index=False)

if not qa_df.empty:
    qa_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_qa_issues.csv', index=False)
if not exclusion_reasons_df.empty:
    exclusion_reasons_df.to_csv(REVIEW_OUTPUT_DIR / 'dataset_review_exclusion_reasons.csv', index=False)

save_grouped_bar(
    by_class_df,
    index_col='class_name',
    value_cols=['raw_object_count', 'kept_object_count'],
    title='Objects by Class (Raw vs Kept)',
    path=review_charts_dir / 'dataset_review_objects_by_class.png',
)

save_grouped_bar(
    pre_split_class_summary_df,
    index_col='class_name',
    value_cols=['annotations_before_pre_split_rules', 'annotations_after_pre_split_rules'],
    title='Pre-split Annotation Totals by Class',
    path=review_charts_dir / 'dataset_review_pre_split_class_summary.png',
)

save_grouped_bar(
    class_boxes_images_df,
    index_col='class_name',
    value_cols=['annotations_boxes', 'images_with_class'],
    title='Class Annotations/Boxes vs Images with Class',
    path=review_charts_dir / 'dataset_review_class_boxes_images.png',
)

images_by_class_df = pre_split_class_summary_df[['class_name', 'primary_images_after_pre_split_rules']].copy()
images_by_class_df = images_by_class_df.rename(columns={'primary_images_after_pre_split_rules': 'image_count'})
save_bar_chart(
    images_by_class_df,
    x_col='class_name',
    y_col='image_count',
    title='Images by Class (After Pre-split Rules)',
    path=review_charts_dir / 'dataset_review_images_by_class.png',
    sort_desc=True,
)

image_keep_counts = images_df['is_kept'].map({True: 'kept', False: 'excluded'}).value_counts().rename_axis('status').reset_index(name='image_count')
save_bar_chart(
    image_keep_counts,
    x_col='status',
    y_col='image_count',
    title='Images Kept vs Excluded',
    path=review_charts_dir / 'dataset_review_images_kept_vs_excluded.png',
)

save_hist(
    images_df['kept_object_count'],
    title='Kept Object Count per Image',
    path=review_charts_dir / 'dataset_review_images_kept_object_hist.png',
    bins=25,
)

if not matching_summary_df.empty:
    matching_plot = matching_summary_df.copy()
    matching_plot['match_group'] = matching_plot['match_status'].astype(str) + ' | ' + matching_plot['match_mode'].astype(str)
    save_bar_chart(
        matching_plot,
        x_col='match_group',
        y_col='image_count',
        title='Matching Summary',
        path=review_charts_dir / 'dataset_review_matching_summary.png',
        sort_desc=True,
    )
else:
    save_no_data_chart(review_charts_dir / 'dataset_review_matching_summary.png', 'Matching Summary')

if not exclusion_reasons_df.empty:
    save_bar_chart(
        exclusion_reasons_df,
        x_col='exclusion_reason',
        y_col='image_count',
        title='Exclusion Reasons',
        path=review_charts_dir / 'dataset_review_exclusion_reasons.png',
        sort_desc=True,
    )
else:
    save_no_data_chart(review_charts_dir / 'dataset_review_exclusion_reasons.png', 'Exclusion Reasons')

if not qa_df.empty:
    save_bar_chart(
        qa_df,
        x_col='qa_issue',
        y_col='count',
        title='QA Issues',
        path=review_charts_dir / 'dataset_review_qa_issues.png',
        sort_desc=True,
    )
else:
    save_no_data_chart(review_charts_dir / 'dataset_review_qa_issues.png', 'QA Issues')

rename_kind_counts = rename_plan_df.groupby('kind').size().rename('count').reset_index() if not rename_plan_df.empty else pd.DataFrame(columns=['kind', 'count'])
save_bar_chart(
    rename_kind_counts,
    x_col='kind',
    y_col='count',
    title='Rename Plan by Kind',
    path=review_charts_dir / 'dataset_review_rename_plan_kind.png',
)

orphan_count_df = pd.DataFrame([{'metric': 'orphan_labels', 'count': int(len(orphan_labels_df))}])
save_bar_chart(
    orphan_count_df,
    x_col='metric',
    y_col='count',
    title='Orphan Label Count',
    path=review_charts_dir / 'dataset_review_orphan_labels_count.png',
)

if not pre_split_undersample_actions_df.empty:
    save_grouped_bar(
        pre_split_undersample_actions_df,
        index_col='class_name',
        value_cols=['annotations_before', 'annotations_target_cap', 'annotations_dropped'],
        title='Pre-split Undersample Actions',
        path=review_charts_dir / 'dataset_review_pre_split_undersample_actions.png',
    )
else:
    save_no_data_chart(
        review_charts_dir / 'dataset_review_pre_split_undersample_actions.png',
        'Pre-split Undersample Actions',
    )

review_summary = {
    'images_total': int(len(records)),
    'images_kept': int(len(kept_records)),
    'images_excluded': int(len(excluded_records)),
    'classes_total': int(len(classes)),
    'excluded_class_ids_manual': sorted([int(x) for x in excluded_class_ids]),
    'excluded_class_ids_auto_pre_split': sorted([int(x) for x in pre_split_auto_excluded_class_ids]),
    'excluded_class_ids_effective': sorted([int(x) for x in effective_excluded_class_ids]),
    'pre_split_exclude_below_annotations': PRE_SPLIT_EXCLUDE_BELOW_ANNOTATIONS,
    'pre_split_undersample_enabled': ENABLE_PRE_SPLIT_UNDERSAMPLE,
    'pre_split_undersample_cap': pre_split_undersample_cap,
    'pre_split_undersample_actions_count': int(len(pre_split_undersample_actions_df)),
    'orphan_labels': int(len(orphan_label_paths)),
    'rename_plan_count': int(len(rename_plan)),
    'apply_inplace_rename': APPLY_INPLACE_RENAME,
    'dry_run_rename': DRY_RUN_RENAME,
    'qa_issue_counts': {k: int(v) for k, v in qa_counters.items()},
    'charts_dir': str(review_charts_dir),
}

(REVIEW_OUTPUT_DIR / 'dataset_review_summary.json').write_text(
    json.dumps(review_summary, indent=2),
    encoding='utf-8',
)

print({'review_output_dir': str(REVIEW_OUTPUT_DIR), 'review_charts_dir': str(review_charts_dir)})

{'review_output_dir': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset_review_and_split\\review', 'review_charts_dir': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset_review_and_split\\review\\charts'}


In [72]:
rng = random.Random(RANDOM_SEED)
split_names = list(SPLIT_RATIOS.keys())

if STRATIFY_BY_PRIMARY_CLASS:
    strata: dict[str, list[ImageRecord]] = {}
    for rec in kept_records:
        if rec.kept_class_ids:
            primary_class_id = Counter(rec.kept_class_ids).most_common(1)[0][0]
            stratum_key = class_name_by_id.get(primary_class_id, f'UNKNOWN_{primary_class_id}')
        else:
            stratum_key = '__NO_OBJECTS__'
        strata.setdefault(stratum_key, []).append(rec)
else:
    strata = {'ALL_IMAGES': list(kept_records)}

assignments = {name: [] for name in split_names}

for stratum_key, stratum_records in sorted(strata.items()):
    local = list(stratum_records)
    rng.shuffle(local)

    counts = split_counts(len(local), SPLIT_RATIOS)
    start = 0
    for split_name in split_names:
        n = counts[split_name]
        assignments[split_name].extend(local[start:start + n])
        start += n

for split_name in split_names:
    assignments[split_name] = sorted(assignments[split_name], key=lambda r: r.image_name)

print({k: len(v) for k, v in assignments.items()})

{'train': 633, 'val': 78, 'test': 76}


In [73]:
if CLEAN_OUTPUT_BEFORE_WRITE and SPLIT_OUTPUT_DIR.exists():
    shutil.rmtree(SPLIT_OUTPUT_DIR)

for split_name in split_names:
    (SPLIT_OUTPUT_DIR / 'images' / split_name).mkdir(parents=True, exist_ok=True)
    (SPLIT_OUTPUT_DIR / 'labels' / split_name).mkdir(parents=True, exist_ok=True)

split_rows = []
split_object_counter = Counter()

def remap_label_lines_for_training(label_lines: list[str], id_map: dict[int, int]) -> list[str]:
    remapped = []
    for line in label_lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        old_id = int(float(parts[0]))
        if old_id not in id_map:
            continue
        new_id = id_map[old_id]
        remapped.append(' '.join([str(new_id), *parts[1:]]))
    return remapped

for split_name in split_names:
    for rec in assignments[split_name]:
        dst_image = SPLIT_OUTPUT_DIR / 'images' / split_name / rec.image_name
        dst_label = SPLIT_OUTPUT_DIR / 'labels' / split_name / f'{Path(rec.image_name).stem}.txt'

        if COPY_IMAGES:
            shutil.copy2(rec.image_path, dst_image)

        remapped_label_lines = remap_label_lines_for_training(rec.kept_label_lines, final_class_id_remap)

        if WRITE_FILTERED_LABELS:
            if remapped_label_lines:
                dst_label.write_text('\n'.join(remapped_label_lines) + '\n', encoding='utf-8')
            elif WRITE_EMPTY_LABEL_FILES:
                dst_label.write_text('', encoding='utf-8')

        for cid in rec.kept_class_ids:
            if cid in final_class_id_remap:
                split_object_counter[(split_name, final_class_id_remap[cid])] += 1

        split_rows.append({
            'split': split_name,
            'image_name': rec.image_name,
            'image_path_source': str(rec.image_path),
            'label_path_source': str(rec.label_path) if rec.label_path else '',
            'image_path_dest': str(dst_image) if COPY_IMAGES else '',
            'label_path_dest': str(dst_label) if WRITE_FILTERED_LABELS else '',
            'raw_object_count': rec.raw_object_count,
            'kept_object_count': rec.kept_object_count,
            'excluded_object_count': rec.excluded_object_count,
            'seed': RANDOM_SEED,
        })

split_manifest_df = pd.DataFrame(split_rows)

split_image_summary_df = (
    split_manifest_df.groupby('split').size().rename('image_count').reset_index()
    if not split_manifest_df.empty
    else pd.DataFrame(columns=['split', 'image_count'])
)

split_object_summary_df = pd.DataFrame([
    {
        'split': split_name,
        'class_id': cid,
        'class_name': final_class_names[cid],
        'source_class_id': int(final_class_id_unmap[cid]),
        'object_count': count,
    }
    for (split_name, cid), count in sorted(split_object_counter.items())
])

SPLIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
split_manifest_df.to_csv(SPLIT_OUTPUT_DIR / 'split_manifest.csv', index=False)
split_image_summary_df.to_csv(SPLIT_OUTPUT_DIR / 'split_image_summary.csv', index=False)
split_object_summary_df.to_csv(SPLIT_OUTPUT_DIR / 'split_object_summary.csv', index=False)

pair_rows = []
pair_errors = []

for split_name in split_names:
    split_images_dir = SPLIT_OUTPUT_DIR / 'images' / split_name
    split_labels_dir = SPLIT_OUTPUT_DIR / 'labels' / split_name

    image_stems = {p.stem for p in split_images_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS}
    label_paths_in_split = [p for p in split_labels_dir.iterdir() if p.is_file() and p.suffix.lower() == LABEL_EXTENSION.lower()]
    label_stems = {p.stem for p in label_paths_in_split}

    missing_label_stems = sorted(image_stems - label_stems)
    orphan_label_stems = sorted(label_stems - image_stems)

    if FORCE_LABEL_FILE_FOR_EVERY_IMAGE:
        for stem in missing_label_stems:
            (split_labels_dir / f'{stem}{LABEL_EXTENSION}').write_text('', encoding='utf-8')

    if REMOVE_ORPHAN_LABEL_FILES:
        for stem in orphan_label_stems:
            (split_labels_dir / f'{stem}{LABEL_EXTENSION}').unlink(missing_ok=True)

    image_stems = {p.stem for p in split_images_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS}
    label_stems = {p.stem for p in split_labels_dir.iterdir() if p.is_file() and p.suffix.lower() == LABEL_EXTENSION.lower()}

    missing_after_fix = sorted(image_stems - label_stems)
    orphan_after_fix = sorted(label_stems - image_stems)

    pair_rows.append({
        'split': split_name,
        'images': len(image_stems),
        'labels': len(label_stems),
        'missing_labels_after_fix': len(missing_after_fix),
        'orphan_labels_after_fix': len(orphan_after_fix),
    })

    if missing_after_fix or orphan_after_fix:
        pair_errors.append({
            'split': split_name,
            'missing_labels_after_fix': missing_after_fix[:20],
            'orphan_labels_after_fix': orphan_after_fix[:20],
        })

pair_check_df = pd.DataFrame(pair_rows)
pair_check_df.to_csv(SPLIT_OUTPUT_DIR / 'split_pair_check.csv', index=False)

if pair_errors:
    (SPLIT_OUTPUT_DIR / 'split_pair_check_errors.json').write_text(json.dumps(pair_errors, indent=2), encoding='utf-8')

if STRICT_PAIR_CHECK and pair_errors:
    raise RuntimeError('Image/label mismatch remains after post-processing. See split_pair_check_errors.json.')

data_yaml_lines = [
    'train: images/train',
    'val: images/val',
    'test: images/test',
    '',
    f'nc: {len(final_class_names)}',
    f"names: {final_class_names}"
]
(SPLIT_OUTPUT_DIR / 'data.yaml').write_text('\n'.join(data_yaml_lines) + '\n', encoding='utf-8')

split_charts_dir = SPLIT_OUTPUT_DIR / 'charts'
split_charts_dir.mkdir(parents=True, exist_ok=True)

def save_no_data_chart(path: Path, title: str, message: str = 'No data') -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, message, ha='center', va='center', fontsize=12)
    ax.set_title(title)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_bar_chart(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    path: Path,
    rotate: int = 45,
) -> None:
    if df.empty or x_col not in df.columns or y_col not in df.columns:
        save_no_data_chart(path, title)
        return

    plot_df = df[[x_col, y_col]].dropna()
    if plot_df.empty:
        save_no_data_chart(path, title)
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df[x_col].astype(str), pd.to_numeric(plot_df[y_col], errors='coerce').fillna(0).astype(float))
    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.tick_params(axis='x', labelrotation=rotate)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_grouped_bar(
    df: pd.DataFrame,
    index_col: str,
    value_cols: list[str],
    title: str,
    path: Path,
    rotate: int = 45,
) -> None:
    if df.empty or index_col not in df.columns or any(col not in df.columns for col in value_cols):
        save_no_data_chart(path, title)
        return

    plot_df = df[[index_col, *value_cols]].copy()
    if plot_df.empty:
        save_no_data_chart(path, title)
        return

    plot_df = plot_df.set_index(index_col)
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_df.plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.set_xlabel(index_col)
    ax.set_ylabel('count')
    ax.tick_params(axis='x', labelrotation=rotate)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_hist(values: pd.Series, title: str, path: Path, bins: int = 30) -> None:
    numeric = pd.to_numeric(values, errors='coerce').dropna()
    if numeric.empty:
        save_no_data_chart(path, title)
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(numeric, bins=bins)
    ax.set_title(title)
    ax.set_xlabel('value')
    ax.set_ylabel('frequency')
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

save_bar_chart(
    split_image_summary_df,
    x_col='split',
    y_col='image_count',
    title='Split Image Counts',
    path=split_charts_dir / 'split_image_summary.png',
    rotate=0,
)

if not split_object_summary_df.empty:
    split_object_pivot = (
        split_object_summary_df
        .pivot_table(index='class_name', columns='split', values='object_count', aggfunc='sum', fill_value=0)
        .reset_index()
    )
    save_grouped_bar(
        split_object_pivot,
        index_col='class_name',
        value_cols=[col for col in split_object_pivot.columns if col != 'class_name'],
        title='Split Object Counts by Class',
        path=split_charts_dir / 'split_object_summary.png',
    )
else:
    save_no_data_chart(split_charts_dir / 'split_object_summary.png', 'Split Object Counts by Class')

pair_plot_df = pair_check_df.copy()
if not pair_plot_df.empty:
    pair_plot_df['pair_status'] = pair_plot_df.apply(
        lambda row: f"{row['split']} (miss:{row['missing_labels_after_fix']}, orphan:{row['orphan_labels_after_fix']})",
        axis=1,
    )
    save_grouped_bar(
        pair_plot_df,
        index_col='pair_status',
        value_cols=['images', 'labels'],
        title='Pair Check (Images vs Labels)',
        path=split_charts_dir / 'split_pair_check.png',
        rotate=10,
    )
else:
    save_no_data_chart(split_charts_dir / 'split_pair_check.png', 'Pair Check (Images vs Labels)')

save_hist(
    split_manifest_df['kept_object_count'] if 'kept_object_count' in split_manifest_df.columns else pd.Series(dtype=float),
    title='Split Manifest Kept Object Count Histogram',
    path=split_charts_dir / 'split_manifest_kept_object_hist.png',
    bins=25,
)

run_config = {
    'dataset_root': str(DATASET_ROOT),
    'images_dir': str(IMAGES_DIR),
    'labels_dir': str(LABELS_DIR),
    'output_root': str(OUTPUT_ROOT),
    'review_output_dir': str(REVIEW_OUTPUT_DIR),
    'split_output_dir': str(SPLIT_OUTPUT_DIR),
    'split_charts_dir': str(split_charts_dir),
    'exclude_class_names': EXCLUDE_CLASS_NAMES,
    'exclude_class_ids_manual': sorted([int(x) for x in excluded_class_ids]),
    'exclude_class_ids_auto_pre_split': sorted([int(x) for x in pre_split_auto_excluded_class_ids]),
    'exclude_class_ids_effective': sorted([int(x) for x in effective_excluded_class_ids]),
    'final_class_ids_source_order': [int(x) for x in final_class_ids],
    'final_class_names_training_order': final_class_names,
    'final_class_id_remap': {str(old): int(new) for old, new in final_class_id_remap.items()},
    'pre_split_exclude_below_annotations': PRE_SPLIT_EXCLUDE_BELOW_ANNOTATIONS,
    'enable_pre_split_undersample': ENABLE_PRE_SPLIT_UNDERSAMPLE,
    'pre_split_undersample_reference': PRE_SPLIT_UNDERSAMPLE_REFERENCE,
    'pre_split_undersample_max_annotations': PRE_SPLIT_UNDERSAMPLE_MAX_ANNOTATIONS,
    'pre_split_undersample_cap': pre_split_undersample_cap,
    'require_label_for_image': REQUIRE_LABEL_FOR_IMAGE,
    'drop_images_with_no_objects': DROP_IMAGES_WITH_NO_OBJECTS,
    'min_objects_per_image': MIN_OBJECTS_PER_IMAGE,
    'max_objects_per_image': MAX_OBJECTS_PER_IMAGE,
    'split_ratios': SPLIT_RATIOS,
    'random_seed': RANDOM_SEED,
    'stratify_by_primary_class': STRATIFY_BY_PRIMARY_CLASS,
    'copy_images': COPY_IMAGES,
    'write_filtered_labels': WRITE_FILTERED_LABELS,
    'write_empty_label_files': WRITE_EMPTY_LABEL_FILES,
    'force_label_file_for_every_image': FORCE_LABEL_FILE_FOR_EVERY_IMAGE,
    'remove_orphan_label_files': REMOVE_ORPHAN_LABEL_FILES,
    'strict_pair_check': STRICT_PAIR_CHECK,
}

(SPLIT_OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')

display(split_image_summary_df)
display(split_object_summary_df.sort_values(['split', 'class_id']).reset_index(drop=True))
display(pair_check_df)
print({
    'split_output_dir': str(SPLIT_OUTPUT_DIR),
    'manifest': str(SPLIT_OUTPUT_DIR / 'split_manifest.csv'),
    'data_yaml': str(SPLIT_OUTPUT_DIR / 'data.yaml'),
    'split_charts_dir': str(split_charts_dir),
})

,split,image_count
0,test,76
1,train,633
2,val,78


,split,class_id,class_name,source_class_id,object_count
0,test,0,DISP,1,39
1,test,1,ECMA,2,116
2,test,2,ECST,3,212
3,test,3,ECTH,4,21
4,test,4,HEMA,6,20
5,test,5,TRGR,8,16
6,train,0,DISP,1,391
7,train,1,ECMA,2,971
8,train,2,ECST,3,1610
9,train,3,ECTH,4,226


,split,images,labels,missing_labels_after_fix,orphan_labels_after_fix
0,train,633,633,0,0
1,val,78,78,0,0
2,test,76,76,0,0


{'split_output_dir': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset_review_and_split\\split', 'manifest': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset_review_and_split\\split\\split_manifest.csv', 'data_yaml': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset_review_and_split\\split\\data.yaml', 'split_charts_dir': 'C:\\Users\\michael.akridge\\Desktop\\20260713\\optics-si-special-projects\\urchins\\multi-class\\v1\\dataset_review_and_split\\split\\charts'}


## Usage Notes

1. Edit the configuration cell first (paths, exclusions, thresholds, split ratios, seed).
2. Run cells top-to-bottom once.
3. Review pre-split class totals to decide exclusions/undersampling before final split.
4. Review CSV outputs under `dataset_review_and_split/review` and `dataset_review_and_split/split`.
5. Use generated `dataset_review_and_split/split/data.yaml` for YOLO training (class IDs are remapped to contiguous training order).
6. If you change filters or split settings, rerun from the config cell downward.